In [ ]:
#instalar os pacotes 

!pip install pandas requests beautifulsoup4 openpyxl tqdm
!pip install selenium
!pip install webdriver-manager
!pip install xlrd
!pip install openpyxl


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# importando os pacotes 

import pandas as pd
import requests
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from tqdm import tqdm
import time
from webdriver_manager.chrome import ChromeDriverManager
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service


In [ ]:
# converter para csv

arquivo = r"C:\Users\GG88\OneDrive - PETROBRAS\Área de Trabalho\ebooks xp\PROJETO APLICADO XP\BASE DE DADOS\links_petroleo.xls"

try:
    df = pd.read_excel(arquivo, engine="xlrd")
    print("Arquivo é XLS verdadeiro")

except Exception:

    try:
        df = pd.read_excel(arquivo, engine="openpyxl")
        print("Arquivo é XLSX disfarçado")

    except Exception:

        try:
            df = pd.read_csv(arquivo, encoding="utf-8")
            print("Arquivo é CSV (utf-8)")

        except Exception:
            df = pd.read_csv(arquivo, encoding="latin1")
            print("Arquivo é CSV (latin1)")

# visualizar dados
print(df.head())

# =========================
# GERAR CSV LIMPO
# =========================

saida = r"C:\Users\GG88\OneDrive - PETROBRAS\Área de Trabalho\links_petroleo_convertido.csv"

df.to_csv(saida, index=False, encoding="utf-8-sig")

print("CSV gerado em:")
print(saida)

Arquivo é CSV (utf-8)
  https://bancodeimagens.petrobras.com.br/fotoweb/archives/5015-Explora%C3%A7%C3%A3o-e-Produ%C3%A7%C3%A3o/Folder%203/Dig47467.jpg.info#c=%2Ffotoweb%2Farchives%2F5015-Explora%25C3%25A7%25C3%25A3o-e-Produ%25C3%25A7%25C3%25A3o%2F
0  https://bancodeimagens.petrobras.com.br/fotowe...                                                                                                                                                                              
1  https://bancodeimagens.petrobras.com.br/fotowe...                                                                                                                                                                              
2  https://bancodeimagens.petrobras.com.br/fotowe...                                                                                                                                                                              
3  https://bancodeimagens.petrobras.com.br/fotowe...                  

In [ ]:
# rodar os codigos 

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from tqdm import tqdm

# =====================================
# CSV COM OS LINKS
# =====================================

arquivo = r"C:\Users\GG88\OneDrive - PETROBRAS\Área de Trabalho\ebooks xp\PROJETO APLICADO XP\BASE DE DADOS\links_petroleo_convertido.csv"

df = pd.read_csv(arquivo)

links = df.iloc[:,0].dropna().tolist()

print("Total de links:", len(links))

# =====================================
# CONFIGURAR SELENIUM
# =====================================

options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--disable-gpu")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

wait = WebDriverWait(driver,10)

# =====================================
# LISTAS DE RESULTADOS
# =====================================

codigos = []
descricoes = []
tags_lista = []

# =====================================
# LOOP PRINCIPAL
# =====================================

for i, link in enumerate(tqdm(links)):

    try:

        driver.get(link)

        codigo = wait.until(
            EC.presence_of_element_located((By.ID,"preview-title"))
        ).text.strip()

        descricao = driver.find_element(
            By.CSS_SELECTOR,
            'div.js-metadata[data-builtin="description"] .js-meta-field-inner-content'
        ).text.strip()

        tags = driver.find_elements(
            By.CSS_SELECTOR,
            'div.js-metadata[data-builtin="tags"] a'
        )

        tags_texto = " ".join([t.text.strip() for t in tags])

    except Exception:

        codigo = ""
        descricao = ""
        tags_texto = ""

    codigos.append(codigo)
    descricoes.append(descricao)
    tags_lista.append(tags_texto)

    # =====================================
    # BACKUP A CADA 200 LINKS
    # =====================================

    if (i+1) % 200 == 0:

        backup = pd.DataFrame({
            "link": links[:len(codigos)],
            "codigo": codigos,
            "descricao": descricoes,
            "tags": tags_lista
        })

        backup.to_csv(f"backup_{i+1}.csv", index=False, encoding="utf-8-sig")

        print(f"\nBackup salvo: backup_{i+1}.csv")

# =====================================
# FINALIZAR
# =====================================

driver.quit()

resultado = pd.DataFrame({
    "link": links,
    "codigo": codigos,
    "descricao": descricoes,
    "tags": tags_lista
})

resultado.to_csv("metadados_petroleo.csv", index=False, encoding="utf-8-sig")

print("\nExtração finalizada.")

Total de links: 12200


ConnectionError: Could not reach host. Are you offline?

Esse erro não é do Selenium em si. Ele acontece porque você está em uma rede corporativa (provavelmente da Petrobras) que usa certificado SSL interno.

Por isso o Python não consegue acessar:

https://googlechromelabs.github.io

que é de onde o webdriver_manager baixa o ChromeDriver.

Erro principal:

SSLCertVerificationError: self-signed certificate in certificate chain

Ou seja: proxy corporativo interceptando SSL.

✔ Solução mais simples (recomendada)

Não usar webdriver_manager.
Baixar o ChromeDriver manualmente.

1️⃣ Baixe o ChromeDriver

Abra no navegador:

https://googlechromelabs.github.io/chrome-for-testing/

Baixe a versão igual ao seu Chrome.

Depois extraia:

chromedriver.exe
2️⃣ Coloque aqui

Na mesma pasta do script ou por exemplo:

C:\chromedriver\chromedriver.exe